# Exploring the silver zone, step by step

`silver.run()` chains a handful of small, pure functions (DataFrame in → DataFrame out, no I/O).
This notebook calls each one separately on real bronze data so you can see exactly what each step changes.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # config.yaml's lake.root is relative to the repo root, not this notebook's folder

from finlake import silver, gold, lake

cfg = lake.load_config()

## 1. Load bronze data
This is the raw, messy data already ingested by an earlier pipeline run.

In [ ]:
bronze_df = lake.read_partitioned(cfg, "bronze", "transactions")
print(bronze_df.shape)
bronze_df.head()

## 2. Walk through the cleaning steps one at a time
Same drop as `silver.run()` — partition/ingestion columns aren't part of the cleaning logic.

In [ ]:
df = bronze_df.drop(columns=["_ingested_at", "year", "month"], errors="ignore")
df[["merchant", "category", "status", "currency"]].head()

In [ ]:
df = silver.normalize_strings(df)
df[["merchant", "category", "status", "currency"]].head()

In [ ]:
df = silver.parse_timestamps(df)
df["timestamp"].head()

In [ ]:
before = len(df)
df = silver.deduplicate(df)
print(f"{before - len(df)} duplicate transaction_id rows removed")

In [ ]:
df = silver.fill_missing(df)
df[["merchant", "category"]].isna().sum()

## 3. Validate — see what gets rejected, and why
Nothing is silently dropped: every rejected row keeps a `reject_reason`.

In [ ]:
valid, rejected = silver.validate(df, cfg["quality"])
print(f"valid={len(valid)} rejected={len(rejected)}")
rejected["reject_reason"].value_counts()

## 4. Enrich — derived columns analysts actually use

In [ ]:
silver_df = silver.enrich(valid)
silver_df[["amount_abs", "amount_bucket", "is_weekend", "is_refund"]].head()

## 5. Gold aggregations
Same pattern — pure functions, this time taking the silver DataFrame straight to business-ready tables.

In [ ]:
gold.monthly_summary(silver_df)

In [ ]:
gold.top_merchants(silver_df)

## 6. Quick chart

In [ ]:
gold.monthly_summary(silver_df).plot(x="month", y="total_spend", kind="bar", title="Monthly spend", legend=False)